In [2]:
from pathlib import Path
from pysdmx.api.fmr import RegistryClient
from pysdmx.io import get_datasets
from vtlengine import run_sdmx

In [3]:
AGRICULTURE_DATA_QUERY = "https://sdmx.oecd.org/public/rest/data/OECD.TAD.ATM,DSD_AGR@DF_OUTLOOK_2021_2030,1.0/OECD.A....?startPeriod=2024&endPeriod=2026"
AGRICULTURE_METADATA_QUERY = "https://sdmx.oecd.org/public/rest/dataflow/OECD.TAD.ATM/DSD_AGR@DF_OUTLOOK_2021_2030/1.0?references=all"

agriculture_data = get_datasets(
    data=AGRICULTURE_DATA_QUERY,
    structure=AGRICULTURE_METADATA_QUERY)

POPULATION_DATA_QUERY = "https://sdmx.oecd.org/public/rest/data/OECD.ELS.SAE,DSD_POPULATION@DF_POP_HIST,1.0/.POP.PS._T._T.?startPeriod=2024"
POPULATION_METADATA_QUERY = "https://sdmx.oecd.org/public/rest/dataflow/OECD.ELS.SAE/DSD_POPULATION@DF_POP_HIST/1.0?references=all"

population_data = get_datasets(
    data=POPULATION_DATA_QUERY,
    structure=POPULATION_METADATA_QUERY)

In [4]:
FMR_ENDPOINT = "https://fmr.meaningfuldata.eu/sdmx/v2"

VALIDATION_QUERY = {
        'id': 'OECD_AGRIGULTURE_VALIDATIONS',
        'agency': 'MD',
        'version': '1.0',
        'api_endpoint': FMR_ENDPOINT
    }

DERIVATION_QUERY = {
        'id': 'OECD_AGRICULTURE_DERIVATION',
        'agency': 'MD',
        'version': '1.0',
        'api_endpoint': FMR_ENDPOINT
    }

def get_vtl_script(query):
    rc = RegistryClient(api_endpoint=query['api_endpoint'])

    vtl_transformation_scheme = rc.get_vtl_transformation_scheme(
        id=query['id'],
        agency=query['agency'],
        version=query['version'])
    
    return vtl_transformation_scheme

validation_script = get_vtl_script(VALIDATION_QUERY)
calculation_script = get_vtl_script(DERIVATION_QUERY)


https://fmr.meaningfuldata.eu/sdmx/v2/structure/transformationscheme/MD/OECD_AGRIGULTURE_VALIDATIONS/1.0?detail=referencepartial&references=descendants
https://fmr.meaningfuldata.eu/sdmx/v2/structure/transformationscheme/MD/OECD_AGRICULTURE_DERIVATION/1.0?detail=referencepartial&references=descendants


In [5]:
def run_vtl_script(vtl_script, datasets, mappings=None, output_folder=None):


    result = run_sdmx(vtl_script, datasets, mappings, return_only_persistent=True)

    if output_folder:
        for key, value in result.items():
            value.data.to_csv(output_folder / f"{key}_logs.csv", index=False)

    return result

In [6]:
VALIDATION_OUTPUT_FOLDER = Path().cwd().parent / "output" / "validations"

validations_result = run_vtl_script(validation_script, agriculture_data, output_folder=VALIDATION_OUTPUT_FOLDER)

In [7]:
DERIVATION_OUTPUT_FOLDER = Path().cwd().parent / "output" / "derivation"

input_datasets = agriculture_data + population_data


mappings = {
    population_data[0].structure.short_urn: 'DSD_POP',
    agriculture_data[0].structure.short_urn: 'DSD_AGR'
    }

derivation_result = run_vtl_script(calculation_script, input_datasets, mappings=mappings, output_folder=DERIVATION_OUTPUT_FOLDER)